In [1]:
import re
import csv
from pathlib import Path
from typing import List, Optional, Tuple


def _remove_latex_comments(tex: str) -> str:
    """
    Remove LaTeX comments. Anything after an unescaped % is a comment.
    (i.e., \% is kept)
    """
    return re.sub(r'(?<!\\)%.*', '', tex)


def _consume_balanced_braces(s: str, i: int) -> int:
    """Given s[i] == '{', return index just after the matching '}' (supports nesting)."""
    if i >= len(s) or s[i] != '{':
        raise ValueError("Expected '{' at position i")
    depth = 0
    j = i
    while j < len(s):
        if s[j] == '{':
            depth += 1
        elif s[j] == '}':
            depth -= 1
            if depth == 0:
                return j + 1
        j += 1
    raise ValueError("Unbalanced braces in tabular arguments.")

def _strip_begin_args_balanced(env: str, body: str) -> str:
    """
    Strip the arguments right after \\begin{tabularx/tabular} ...:
      - tabularx: optional [..], then {width}{colspec}
      - tabular:  optional [..], then {colspec}
    Works with nested braces in colspec like *{10}{Y}.
    """
    s = body.lstrip()
    i = 0

    # optional [..]
    if i < len(s) and s[i] == '[':
        k = s.find(']')
        if k == -1:
            raise ValueError("Unbalanced [...] in tabular arguments.")
        i = k + 1
        while i < len(s) and s[i].isspace():
            i += 1

    # consume required brace args
    need = 2 if env == "tabularx" else 1
    for _ in range(need):
        while i < len(s) and s[i].isspace():
            i += 1
        if i >= len(s) or s[i] != '{':
            raise ValueError("Tabular arguments not found where expected.")
        i = _consume_balanced_braces(s, i)

    return s[i:].lstrip()

def _extract_first_tabular(tex: str) -> Tuple[str, str]:
    """
    Extract first tabular/tabularx and return (env, inner_content_with_args_stripped).
    """
    m = re.search(
        r'\\begin\{(tabularx|tabular)\}(.+?)\\end\{\1\}',
        tex,
        flags=re.DOTALL
    )
    if not m:
        raise ValueError("No tabular/tabularx environment found.")

    env = m.group(1)
    body = m.group(2).strip()

    # robustly strip begin-args (handles nested braces)
    body = _strip_begin_args_balanced(env, body)
    return env, body

def _strip_row_level_commands(s: str) -> str:
    # Remove common rule/line commands that can appear near rows
    s = re.sub(r'\\(toprule|midrule|bottomrule)\b', '', s)
    s = re.sub(r'\\cmidrule\b(\([^)]*\))?\{[^}]*\}', '', s)
    s = re.sub(r'\\hline\b', '', s)
    return s.strip()


def _clean_cell(cell: str) -> str:
    t = cell.strip()

    # Keep content for multirow/multicolumn
    t = re.sub(r'\\multirow\{[^}]*\}\{[^}]*\}\{([^}]*)\}', r'\1', t)
    t = re.sub(r'\\multicolumn\{[^}]*\}\{[^}]*\}\{([^}]*)\}', r'\1', t)

    # Unwrap common formatting commands (repeat to handle simple nesting)
    for _ in range(4):
        t = re.sub(r'\\[a-zA-Z]+\*?(?:\[[^\]]*\])?\{([^{}]*)\}', r'\1', t)

    # Remove inline math dollars if present
    t = re.sub(r'\$([^$]*)\$', r'\1', t)

    # Unescape common latex escapes
    t = t.replace(r'\&', '&').replace(r'\%', '%').replace(r'\_', '_')

    # Drop any remaining braces
    t = t.replace('{', '').replace('}', '')

    # Normalize whitespace
    t = re.sub(r'\s+', ' ', t).strip()
    return t


def latex_table_to_rows(latex_text: str, fill_down: bool = True) -> List[List[str]]:
    """
    Parse the first tabular/tabularx in latex_text into a 2D list (rows x cols).
    - Ignores comments after %.
    - Skips non-data lines (e.g., \\toprule, \\cmidrule).
    - Strips formatting like \\textbf{}, \\underline{}, \\multirow{} (keeps content).
    - If fill_down=True, fills empty cells with the value from the previous row (useful for \\multirow tables).
    """
    tex = _remove_latex_comments(latex_text)
    _, body = _extract_first_tabular(tex)

    # Split rows on LaTeX row terminator '\\'
    raw_rows = re.split(r'\\\\', body)

    rows: List[List[str]] = []
    for r in raw_rows:
        r = _strip_row_level_commands(r)
        if not r:
            continue
        if '&' not in r:
            continue  # skip lines without table cells
        cells = [_clean_cell(c) for c in r.split('&')]
        # Remove trailing empty cells caused by stray separators (rare)
        while cells and cells[-1] == '':
            cells.pop()
        rows.append(cells)

    if not rows:
        return rows

    # Normalize column counts (pad shorter rows)
    max_cols = max(len(r) for r in rows)
    for r in rows:
        if len(r) < max_cols:
            r.extend([''] * (max_cols - len(r)))

    # Fill down empty cells (skip header row)
    if fill_down and len(rows) >= 2:
        prev = rows[1].copy()
        for i in range(2, len(rows)):
            for j in range(max_cols):
                if rows[i][j] == '' and prev[j] != '':
                    rows[i][j] = prev[j]
            prev = rows[i].copy()

    return rows


def latex_table_to_csv(latex_text: str, csv_path: str, fill_down: bool = True, encoding: str = "utf-8") -> None:
    rows = latex_table_to_rows(latex_text, fill_down=fill_down)
    out = Path(csv_path)
    out.parent.mkdir(parents=True, exist_ok=True)

    with out.open("w", newline="", encoding=encoding) as f:
        writer = csv.writer(f)
        writer.writerows(rows)

<>:10: SyntaxWarning: invalid escape sequence '\%'
<>:10: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_4055016/3708212866.py:10: SyntaxWarning: invalid escape sequence '\%'
  (i.e., \% is kept)


In [2]:

# # 用法示例：把下面的 latex_str 替换为你的 LaTeX 表格字符串，或从 .tex 文件读取
# latex_str = r"""
# \begin{table*}[ht!]
# % \caption{ACC performance of OSTC on the BOLT benchmark under different KCRs (LAR = 1.0)}
# % \label{table::openset_11}
# \centering
# \tiny
# \setlength{\tabcolsep}{4pt}

# % Y 列型：可在此处定义（若已在导言区定义，可删掉下一行）
# \newcolumntype{Y}{>{\centering\arraybackslash}X}
# \begin{tabularx}{\textwidth}{@{}l l l |l *{10}{Y}|Y@{}}
# \toprule
# Metric & KCR & Method & 20NG & THUCNews & Yahoo & TREC & BANK & S.O. & CLINC & HWU & ECDT & M-CID & DBPedia & \textbf{Avg.} \\ \midrule
# \multirow{48}{*}{ACC} & \multirow{16}{*}{0.25} & DOC & 91.41 & 67.67 & 57.33 & 75.55 & 79.18 & 67.42 & 88.68 & 75.93 & 77.32 & 70.94 & 86.25 & 76.15 \\
#  &  & DeepUNK & \underline{96.85} & 34.31 & 62.46 & 79.63 & 84.52 & 83.54 & 89.29 & \underline{87.56} & 77.43 & \textbf{82.99} & 62.12 & 76.43 \\
#  &  & ADB & 69.48 & 77.96 & 55.02 & 71.41 & 77.34 & 85.91 & 88.46 & 76.51 & 81.78 & 61.75 & 82.89 & 75.32 \\
#  &  & SCL & 87.29 & 57.54 & 63.47 & 79.82 & 69.79 & 65.68 & 72.61 & 61.88 & 10.81 & 78.01 & 66.06 & 64.81 \\
#  &  & AB & 70.54 & 43.18 & 64.79 & 54.54 & 73.43 & 75.91 & 81.80 & 80.30 & 61.10 & 77.04 & 70.76 & 68.49 \\
#  &  & KNNCon & 87.22 & 59.92 & 33.78 & 57.85 & 63.69 & 60.26 & 81.63 & 62.33 & 55.55 & 50.53 & 67.90 & 61.88 \\
#  &  & DyEn & \textbf{97.53} & 64.26 & 26.85 & 66.88 & 66.36 & 53.80 & 85.38 & 68.15 & 78.02 & 63.21 & 67.16 & 67.05 \\
#  &  & LLM-MaxSoftmax & 41.22 & 36.60 & 29.49 & 42.88 & 41.13 & 36.88 & 49.16 & 42.06 & 50.27 & 40.60 & 53.94 & 42.20 \\
#  &  & LLM-OpenMax & 83.93 & 64.94 & 30.89 & 75.47 & 85.86 & 68.29 & 91.86 & 83.58 & 75.96 & 56.25 & 85.07 & 72.92 \\
#  &  & LLM-TempScale & 40.44 & 36.82 & 29.70 & 42.49 & 41.03 & 37.07 & 49.18 & 41.67 & 46.88 & 39.40 & 54.32 & 41.73 \\
#  %&  & Mahalanobis & 64.52 & 56.12 & 51.26 & 63.06 & 69.95 & 48.61 & 45.92 & 63.95 & 56.90 & 62.21 & 68.51 & 59.18 \\
#  &  & LLM-Energy & 91.00 & \textbf{85.38} & \textbf{77.23} & \textbf{85.27} & \textbf{87.00} & \textbf{91.95} & \textbf{94.22} & \textbf{87.82} & \underline{88.39} & \underline{80.36} & \underline{91.88} & \textbf{87.32} \\
#  &  & LLM-LogitNorm & 43.18 & 33.57 & 41.35 & 62.90 & 68.24 & 33.59 & 86.60 & 65.12 & 43.74 & 37.79 & 88.86 & 54.99 \\
#  &  & LLM-Entropy & 44.30 & 39.25 & 38.40 & 44.24 & 37.45 & 38.41 & 46.05 & 41.41 & 53.94 & 46.22 & 51.73 & 43.76 \\
#  &  & LLM-KLMatching & 33.75 & 32.57 & 26.06 & 36.94 & 32.84 & 33.16 & 39.99 & 36.59 & 35.45 & 34.35 & 43.32 & 35.00 \\
#  &  & LLM-MaxLogit & 90.50 & \underline{85.02} & \underline{77.16} & \underline{84.76} & \underline{86.74} & \underline{91.88} & \underline{94.16} & 87.43 & \textbf{88.66} & 80.09 & \textbf{91.93} & \underline{87.12} \\
# \cmidrule(l){2-15}
#  & \multirow{16}{*}{0.5} & DOC & 83.65 & 63.67 & 58.56 & 73.88 & 76.63 & 72.28 & 84.58 & 75.21 & 50.34 & 74.14 & 73.06 & 71.45 \\
#  &  & DeepUNK & 95.93 & 35.82 & 56.46 & 65.35 & 71.85 & 75.71 & 79.34 & 78.31 & 50.49 & 72.69 & 50.92 & 66.62 \\
#  &  & ADB & 82.34 & 83.90 & 47.55 & 78.82 & 79.68 & \textbf{87.07} & 86.93 & 75.29 & \textbf{83.10} & 74.44 & 82.58 & 78.34 \\
#  &  & SCL & 69.66 & 50.77 & 40.29 & 70.60 & 57.85 & 61.12 & 73.85 & 54.28 & 9.64 & 71.36 & 54.76 & 55.83 \\
#  &  & AB & 60.96 & 38.38 & 50.62 & 46.02 & 65.10 & 67.30 & 74.50 & 70.54 & 52.64 & 64.58 & 61.56 & 59.29 \\
#  &  & KNNCon & 89.75 & 72.20 & 49.26 & 69.51 & 67.84 & 75.35 & 81.59 & 67.40 & 80.66 & 68.85 & 71.43 & 72.17 \\
#  &  & DyEn & \underline{97.78} & 71.86 & 42.84 & 70.23 & 70.33 & 65.49 & 82.04 & 63.15 & \underline{81.72} & \underline{74.66} & 72.17 & 72.02 \\
#  &  & LLM-MaxSoftmax & 59.97 & 56.13 & 58.79 & 60.16 & 57.71 & 55.73 & 62.23 & 57.75 & 65.97 & 59.52 & 65.59 & 59.96 \\
#  &  & LLM-OpenMax & 92.63 & 81.24 & 42.48 & 82.12 & 78.04 & 80.48 & 86.95 & 79.01 & 78.03 & 73.84 & 73.62 & 77.13 \\
#  &  & LLM-TempScale & 59.97 & 56.13 & 58.18 & 60.16 & 57.71 & 55.97 & 62.24 & 57.75 & 65.06 & 59.27 & 65.93 & 59.85 \\
#  %&  & Mahalanobis & 42.71 & 48.40 & 42.95 & 59.88 & 52.25 & 47.85 & 70.56 & 47.36 & 45.27 & 48.46 & 55.39 & 51.01 \\
#  &  & LLM-Energy & \textbf{98.08} & \underline{85.78} & \underline{66.63} & \underline{84.08} & \underline{82.03} & 86.34 & 90.45 & \underline{83.45} & 77.71 & 74.26 & 86.12 & \underline{83.18} \\
#  &  & LLM-LogitNorm & 74.74 & 62.49 & 50.23 & 81.80 & 81.38 & 64.99 & \underline{90.93} & 80.99 & 80.62 & 69.00 & \textbf{88.57} & 75.07 \\
#  &  & LLM-Entropy & 61.12 & 56.28 & 64.31 & 58.45 & 54.46 & 54.95 & 59.89 & 55.99 & 66.42 & 62.05 & 62.92 & 59.71 \\
#  &  & LLM-KLMatching & 56.97 & 54.24 & 43.88 & 58.20 & 54.66 & 54.10 & 57.69 & 54.86 & 58.47 & 56.68 & 58.32 & 55.28 \\
#  &  & LLM-MaxLogit & 96.13 & \textbf{86.04} & \textbf{68.83} & \textbf{84.94} & \textbf{83.10} & \underline{86.79} & \textbf{91.48} & \textbf{84.55} & 79.53 & \textbf{75.71} & \underline{88.07} & \textbf{84.11} \\
# \cmidrule(l){2-15}
#  & \multirow{16}{*}{0.75} & DOC & 79.47 & 51.25 & 42.55 & 71.84 & 70.99 & 75.56 & 81.40 & 73.55 & 28.10 & 67.67 & 61.33 & 63.97 \\
#  &  & DeepUNK & 95.55 & 37.38 & 47.11 & 45.39 & 53.99 & 68.99 & 68.12 & 68.84 & 28.31 & 61.03 & 36.75 & 55.59 \\
#  &  & ADB & 89.09 & \textbf{83.59} & 59.09 & 80.86 & \textbf{81.01} & \underline{83.05} & \underline{88.36} & 77.21 & \underline{84.61} & 73.78 & \underline{85.69} & \underline{80.58} \\
#  &  & SCL & 80.20 & 39.54 & 37.82 & 72.27 & 60.47 & 61.65 & 78.63 & 64.37 & 13.30 & 69.39 & 64.69 & 58.39 \\
#  &  & AB & 58.40 & 32.54 & 46.48 & 43.06 & 63.50 & 63.72 & 72.28 & 66.94 & 49.52 & 58.60 & 60.26 & 55.94 \\
#  &  & KNNCon & 94.86 & 79.70 & 59.82 & 81.65 & 79.67 & 81.31 & 88.26 & \underline{78.00} & 83.78 & \textbf{79.42} & 77.89 & 80.40 \\
#  &  & DyEn & 94.06 & 76.59 & 57.03 & \underline{81.88} & 78.91 & 76.31 & 87.11 & 76.75 & \textbf{88.46} & \underline{78.79} & 82.44 & 79.85 \\
#  &  & LLM-MaxSoftmax & 85.28 & 72.29 & \underline{65.99} & 77.27 & 75.95 & 75.30 & 79.59 & 74.55 & 77.12 & 75.29 & 79.52 & 76.20 \\
#  &  & LLM-OpenMax & 96.16 & 79.55 & 62.62 & 77.31 & 71.37 & \textbf{86.31} & 85.24 & 77.13 & 68.21 & 69.49 & 65.80 & 76.29 \\
#  &  & LLM-TempScale & 85.28 & 72.43 & 65.86 & 77.27 & 75.96 & 75.31 & 79.95 & 74.57 & 76.99 & 75.11 & 79.56 & 76.21 \\
#  %&  & Mahalanobis & 95.15 & 70.16 & 61.91 & 49.63 & 71.77 & 70.82 & 86.03 & 72.93 & 32.99 & 41.09 & 79.01 & 66.50 \\
#  &  & LLM-Energy & \textbf{99.02} & 78.75 & 40.92 & 76.69 & 71.47 & 78.84 & 83.50 & 72.40 & 69.87 & 63.26 & 81.04 & 74.16 \\
#  &  & LLM-LogitNorm & 92.83 & 79.44 & 64.23 & \textbf{84.57} & \underline{79.84} & 82.41 & \textbf{90.64} & \textbf{79.21} & 84.00 & 76.37 & \textbf{86.25} & \textbf{81.80} \\
#  &  & LLM-Entropy & 86.27 & 72.78 & \textbf{66.76} & 76.57 & 74.94 & 74.78 & 78.13 & 74.19 & 77.53 & 75.77 & 78.05 & 75.98 \\
#  &  & LLM-KLMatching & 83.47 & 70.54 & 62.40 & 76.12 & 73.72 & 73.74 & 77.14 & 72.64 & 74.99 & 73.17 & 76.36 & 74.03 \\
#  &  & LLM-MaxLogit & \underline{98.53} & \underline{80.48} & 46.57 & 78.86 & 74.89 & 80.64 & 86.26 & 75.64 & 74.08 & 65.92 & 85.30 & 77.02 \\
# \bottomrule
# \end{tabularx}
# \caption{ACC performance of OSTC on the BOLT benchmark under different KCRs (LAR = 1.0)}
# \label{table::openset_11}
# \end{table*}
#     """

# # 方式1：写出 CSV 文件
# latex_table_to_csv(latex_str, "table_s13.csv", fill_down=True)

# # 方式2：直接拿到二维数组
# rows = latex_table_to_rows(latex_str, fill_down=True)
# print("rows:", len(rows), "cols:", max(len(r) for r in rows) if rows else 0)
# print(rows[0] if rows else [])

In [3]:

# # 用法示例：把下面的 latex_str 替换为你的 LaTeX 表格字符串，或从 .tex 文件读取
# latex_str = r"""
# \begin{table*}[ht!]
# % \caption{ACC performance of OSTC on the BOLT benchmark under different KCRs (LAR = 1.0)}
# % \label{table::openset_11}
# \centering
# \tiny
# \setlength{\tabcolsep}{4pt}

# % Y 列型：可在此处定义（若已在导言区定义，可删掉下一行）
# \newcolumntype{Y}{>{\centering\arraybackslash}X}
# \begin{tabularx}{\textwidth}{@{}l l l |l *{10}{Y}|Y@{}}
# \toprule
# Metric & KCR & Method & 20NG & THUCNews & Yahoo & TREC & BANK & S.O. & CLINC & HWU & ECDT & M-CID & DBPedia & \textbf{Avg.} \\ \midrule
# \multirow{48}{*}{ACC} & \multirow{16}{*}{0.25} & DOC & 91.41 & 67.67 & 57.33 & 75.55 & 79.18 & 67.42 & 88.68 & 75.93 & 77.32 & 70.94 & 86.25 & 76.15 \\
#  &  & DeepUNK & \underline{96.85} & 34.31 & 62.46 & 79.63 & 84.52 & 83.54 & 89.29 & \underline{87.56} & 77.43 & \textbf{82.99} & 62.12 & 76.43 \\
#  &  & ADB & 69.48 & 77.96 & 55.02 & 71.41 & 77.34 & 85.91 & 88.46 & 76.51 & 81.78 & 61.75 & 82.89 & 75.32 \\
#  &  & SCL & 87.29 & 57.54 & 63.47 & 79.82 & 69.79 & 65.68 & 72.61 & 61.88 & 10.81 & 78.01 & 66.06 & 64.81 \\
#  &  & AB & 70.54 & 43.18 & 64.79 & 54.54 & 73.43 & 75.91 & 81.80 & 80.30 & 61.10 & 77.04 & 70.76 & 68.49 \\
#  &  & KNNCon & 87.22 & 59.92 & 33.78 & 57.85 & 63.69 & 60.26 & 81.63 & 62.33 & 55.55 & 50.53 & 67.90 & 61.88 \\
#  &  & DyEn & \textbf{97.53} & 64.26 & 26.85 & 66.88 & 66.36 & 53.80 & 85.38 & 68.15 & 78.02 & 63.21 & 67.16 & 67.05 \\
#  &  & LLM-MaxSoftmax & 41.22 & 36.60 & 29.49 & 42.88 & 41.13 & 36.88 & 49.16 & 42.06 & 50.27 & 40.60 & 53.94 & 42.20 \\
#  &  & LLM-OpenMax & 83.93 & 64.94 & 30.89 & 75.47 & 85.86 & 68.29 & 91.86 & 83.58 & 75.96 & 56.25 & 85.07 & 72.92 \\
#  &  & LLM-TempScale & 40.44 & 36.82 & 29.70 & 42.49 & 41.03 & 37.07 & 49.18 & 41.67 & 46.88 & 39.40 & 54.32 & 41.73 \\
#  %&  & Mahalanobis & 64.52 & 56.12 & 51.26 & 63.06 & 69.95 & 48.61 & 45.92 & 63.95 & 56.90 & 62.21 & 68.51 & 59.18 \\
#  &  & LLM-Energy & 91.00 & \textbf{85.38} & \textbf{77.23} & \textbf{85.27} & \textbf{87.00} & \textbf{91.95} & \textbf{94.22} & \textbf{87.82} & \underline{88.39} & \underline{80.36} & \underline{91.88} & \textbf{87.32} \\
#  &  & LLM-LogitNorm & 43.18 & 33.57 & 41.35 & 62.90 & 68.24 & 33.59 & 86.60 & 65.12 & 43.74 & 37.79 & 88.86 & 54.99 \\
#  &  & LLM-Entropy & 44.30 & 39.25 & 38.40 & 44.24 & 37.45 & 38.41 & 46.05 & 41.41 & 53.94 & 46.22 & 51.73 & 43.76 \\
#  &  & LLM-KLMatching & 33.75 & 32.57 & 26.06 & 36.94 & 32.84 & 33.16 & 39.99 & 36.59 & 35.45 & 34.35 & 43.32 & 35.00 \\
#  &  & LLM-MaxLogit & 90.50 & \underline{85.02} & \underline{77.16} & \underline{84.76} & \underline{86.74} & \underline{91.88} & \underline{94.16} & 87.43 & \textbf{88.66} & 80.09 & \textbf{91.93} & \underline{87.12} \\
# \cmidrule(l){2-15}
#  & \multirow{16}{*}{0.5} & DOC & 83.65 & 63.67 & 58.56 & 73.88 & 76.63 & 72.28 & 84.58 & 75.21 & 50.34 & 74.14 & 73.06 & 71.45 \\
#  &  & DeepUNK & 95.93 & 35.82 & 56.46 & 65.35 & 71.85 & 75.71 & 79.34 & 78.31 & 50.49 & 72.69 & 50.92 & 66.62 \\
#  &  & ADB & 82.34 & 83.90 & 47.55 & 78.82 & 79.68 & \textbf{87.07} & 86.93 & 75.29 & \textbf{83.10} & 74.44 & 82.58 & 78.34 \\
#  &  & SCL & 69.66 & 50.77 & 40.29 & 70.60 & 57.85 & 61.12 & 73.85 & 54.28 & 9.64 & 71.36 & 54.76 & 55.83 \\
#  &  & AB & 60.96 & 38.38 & 50.62 & 46.02 & 65.10 & 67.30 & 74.50 & 70.54 & 52.64 & 64.58 & 61.56 & 59.29 \\
#  &  & KNNCon & 89.75 & 72.20 & 49.26 & 69.51 & 67.84 & 75.35 & 81.59 & 67.40 & 80.66 & 68.85 & 71.43 & 72.17 \\
#  &  & DyEn & \underline{97.78} & 71.86 & 42.84 & 70.23 & 70.33 & 65.49 & 82.04 & 63.15 & \underline{81.72} & \underline{74.66} & 72.17 & 72.02 \\
#  &  & LLM-MaxSoftmax & 59.97 & 56.13 & 58.79 & 60.16 & 57.71 & 55.73 & 62.23 & 57.75 & 65.97 & 59.52 & 65.59 & 59.96 \\
#  &  & LLM-OpenMax & 92.63 & 81.24 & 42.48 & 82.12 & 78.04 & 80.48 & 86.95 & 79.01 & 78.03 & 73.84 & 73.62 & 77.13 \\
#  &  & LLM-TempScale & 59.97 & 56.13 & 58.18 & 60.16 & 57.71 & 55.97 & 62.24 & 57.75 & 65.06 & 59.27 & 65.93 & 59.85 \\
#  %&  & Mahalanobis & 42.71 & 48.40 & 42.95 & 59.88 & 52.25 & 47.85 & 70.56 & 47.36 & 45.27 & 48.46 & 55.39 & 51.01 \\
#  &  & LLM-Energy & \textbf{98.08} & \underline{85.78} & \underline{66.63} & \underline{84.08} & \underline{82.03} & 86.34 & 90.45 & \underline{83.45} & 77.71 & 74.26 & 86.12 & \underline{83.18} \\
#  &  & LLM-LogitNorm & 74.74 & 62.49 & 50.23 & 81.80 & 81.38 & 64.99 & \underline{90.93} & 80.99 & 80.62 & 69.00 & \textbf{88.57} & 75.07 \\
#  &  & LLM-Entropy & 61.12 & 56.28 & 64.31 & 58.45 & 54.46 & 54.95 & 59.89 & 55.99 & 66.42 & 62.05 & 62.92 & 59.71 \\
#  &  & LLM-KLMatching & 56.97 & 54.24 & 43.88 & 58.20 & 54.66 & 54.10 & 57.69 & 54.86 & 58.47 & 56.68 & 58.32 & 55.28 \\
#  &  & LLM-MaxLogit & 96.13 & \textbf{86.04} & \textbf{68.83} & \textbf{84.94} & \textbf{83.10} & \underline{86.79} & \textbf{91.48} & \textbf{84.55} & 79.53 & \textbf{75.71} & \underline{88.07} & \textbf{84.11} \\
# \cmidrule(l){2-15}
#  & \multirow{16}{*}{0.75} & DOC & 79.47 & 51.25 & 42.55 & 71.84 & 70.99 & 75.56 & 81.40 & 73.55 & 28.10 & 67.67 & 61.33 & 63.97 \\
#  &  & DeepUNK & 95.55 & 37.38 & 47.11 & 45.39 & 53.99 & 68.99 & 68.12 & 68.84 & 28.31 & 61.03 & 36.75 & 55.59 \\
#  &  & ADB & 89.09 & \textbf{83.59} & 59.09 & 80.86 & \textbf{81.01} & \underline{83.05} & \underline{88.36} & 77.21 & \underline{84.61} & 73.78 & \underline{85.69} & \underline{80.58} \\
#  &  & SCL & 80.20 & 39.54 & 37.82 & 72.27 & 60.47 & 61.65 & 78.63 & 64.37 & 13.30 & 69.39 & 64.69 & 58.39 \\
#  &  & AB & 58.40 & 32.54 & 46.48 & 43.06 & 63.50 & 63.72 & 72.28 & 66.94 & 49.52 & 58.60 & 60.26 & 55.94 \\
#  &  & KNNCon & 94.86 & 79.70 & 59.82 & 81.65 & 79.67 & 81.31 & 88.26 & \underline{78.00} & 83.78 & \textbf{79.42} & 77.89 & 80.40 \\
#  &  & DyEn & 94.06 & 76.59 & 57.03 & \underline{81.88} & 78.91 & 76.31 & 87.11 & 76.75 & \textbf{88.46} & \underline{78.79} & 82.44 & 79.85 \\
#  &  & LLM-MaxSoftmax & 85.28 & 72.29 & \underline{65.99} & 77.27 & 75.95 & 75.30 & 79.59 & 74.55 & 77.12 & 75.29 & 79.52 & 76.20 \\
#  &  & LLM-OpenMax & 96.16 & 79.55 & 62.62 & 77.31 & 71.37 & \textbf{86.31} & 85.24 & 77.13 & 68.21 & 69.49 & 65.80 & 76.29 \\
#  &  & LLM-TempScale & 85.28 & 72.43 & 65.86 & 77.27 & 75.96 & 75.31 & 79.95 & 74.57 & 76.99 & 75.11 & 79.56 & 76.21 \\
#  %&  & Mahalanobis & 95.15 & 70.16 & 61.91 & 49.63 & 71.77 & 70.82 & 86.03 & 72.93 & 32.99 & 41.09 & 79.01 & 66.50 \\
#  &  & LLM-Energy & \textbf{99.02} & 78.75 & 40.92 & 76.69 & 71.47 & 78.84 & 83.50 & 72.40 & 69.87 & 63.26 & 81.04 & 74.16 \\
#  &  & LLM-LogitNorm & 92.83 & 79.44 & 64.23 & \textbf{84.57} & \underline{79.84} & 82.41 & \textbf{90.64} & \textbf{79.21} & 84.00 & 76.37 & \textbf{86.25} & \textbf{81.80} \\
#  &  & LLM-Entropy & 86.27 & 72.78 & \textbf{66.76} & 76.57 & 74.94 & 74.78 & 78.13 & 74.19 & 77.53 & 75.77 & 78.05 & 75.98 \\
#  &  & LLM-KLMatching & 83.47 & 70.54 & 62.40 & 76.12 & 73.72 & 73.74 & 77.14 & 72.64 & 74.99 & 73.17 & 76.36 & 74.03 \\
#  &  & LLM-MaxLogit & \underline{98.53} & \underline{80.48} & 46.57 & 78.86 & 74.89 & 80.64 & 86.26 & 75.64 & 74.08 & 65.92 & 85.30 & 77.02 \\
# \bottomrule
# \end{tabularx}
# \caption{ACC performance of OSTC on the BOLT benchmark under different KCRs (LAR = 1.0)}
# \label{table::openset_11}
# \end{table*}
#     """

# # 方式1：写出 CSV 文件
# latex_table_to_csv(latex_str, "table_s13.csv", fill_down=True)

# # 方式2：直接拿到二维数组
# rows = latex_table_to_rows(latex_str, fill_down=True)
# print("rows:", len(rows), "cols:", max(len(r) for r in rows) if rows else 0)
# print(rows[0] if rows else [])


In [9]:
for i in range(2, 14):
    latex_str = open(f"latex_or_csv/openset/openset_table_s{i}.tex").read()

    # 方式1：写出 CSV 文件
    latex_table_to_csv(latex_str, f"table_or_csv/openset/table_s{i}.csv", fill_down=True)

    # 方式2：直接拿到二维数组
    rows = latex_table_to_rows(latex_str, fill_down=True)
    print("rows:", len(rows), "cols:", max(len(r) for r in rows) if rows else 0)
    print(rows[0] if rows else [])


rows: 46 cols: 16
['Metric', 'KCR', 'Method', '20NG', 'THUCNews', 'Yahoo', 'TREC', 'BANK', 'S.O.', 'CLINC', 'HWU', 'ECDT', 'M-CID', 'DBPedia', 'X-Topic', 'Avg.']
rows: 46 cols: 16
['Metric', 'KCR', 'Method', '20NG', 'THUCNews', 'Yahoo', 'TREC', 'BANK', 'S.O.', 'CLINC', 'HWU', 'ECDT', 'M-CID', 'DBPedia', 'X-Topic', 'Avg.']
rows: 46 cols: 16
['Metric', 'KCR', 'Method', '20NG', 'THUCNews', 'Yahoo', 'TREC', 'BANK', 'S.O.', 'CLINC', 'HWU', 'ECDT', 'M-CID', 'DBPedia', 'X-Topic', 'Avg.']
rows: 46 cols: 16
['Metric', 'KCR', 'Method', '20NG', 'THUCNews', 'Yahoo', 'TREC', 'BANK', 'S.O.', 'CLINC', 'HWU', 'ECDT', 'M-CID', 'DBPedia', 'X-Topic', 'Avg.']
rows: 46 cols: 16
['Metric', 'KCR', 'Method', '20NG', 'THUCNews', 'Yahoo', 'TREC', 'BANK', 'S.O.', 'CLINC', 'HWU', 'ECDT', 'M-CID', 'DBPedia', 'X-Topic', 'Avg.']
rows: 46 cols: 16
['Metric', 'KCR', 'Method', '20NG', 'THUCNews', 'Yahoo', 'TREC', 'BANK', 'S.O.', 'CLINC', 'HWU', 'ECDT', 'M-CID', 'DBPedia', 'X-Topic', 'Avg.']
rows: 46 cols: 16
['Metric',